In [ ]:
%pip install -q ipylab

# Plugins

Plugins are povided by the [pluggy](https://pluggy.readthedocs.io/en/stable/index.html#pluggy) plugin system.

Plugins can be registered with the `plugin_manager` which is available as a property of the same name on all objects that subclass from Ipylab.

Implementations of plugins can be reistered with the decorator `@ipylab.hookimpl`.

In [ ]:
# Existing hook specs
from IPython import display as ipd

import ipylab

display(ipd.Markdown("## Plugins\n\nThe following plugins (*hookspecs*) are available."))
for n in dir(ipylab.hookspecs.IpylabHookspec):
    if not n.startswith("_"):
        f = getattr(ipylab.hookspecs.IpylabHookspec, n)
        display(ipd.Markdown(f"### `{f.__name__}`"))
        display(ipd.Markdown(f.__doc__))

## Autostart

Ipylab automatically starts a Python kernel with the path 'ipylab' when the ipylab plugin is activated. The `autostart` hookspec is called as soon as comms has been established. Autostart is performed prior to restoring Jupyterlab such as when refreshing the page or starting from scratch.

Possible uses include:
* Create and register custom commands;
* Create launchers;
* Modify the appearance of Jupyterlab.

### Entry points

Entry points allow for a module to advertise to Ipylab that it provides plugins.

Add the following in your `pyproject.toml`

``` toml
[project.entry-points.ipylab]
myproject = "myproject.pluginmodule:object"
```

In `myproject.pluginmodule.py` Define the plugins.

Example:

```python
# @ipylab_plugin.py

import asyncio

async def startup():
    import ipylab
     
    #Do everything to startup

pluginmodule = None # The module name here should be the same as above.
asyncio.create_task(startup())
```

### Example launching a small app

In [ ]:
# @myproject.pluginmodule.py
async def create_app(cwd, path, cid, app: ipylab.JupyterFrontEnd):
    # The code in this function is called in the new kernel.
    # Ensure imports are performed inside the function.
    import ipywidgets as ipw

    import ipylab

    sc = ipylab.ShellConnection(cid=cid)  # The Main area widget
    panel = ipylab.Panel()
    panel.title.label = path
    panel.title.caption = panel.app.kernelId
    error_button = ipw.Button(
        description="Do an error",
        tooltip="An error dialog will pop up when this is clicked.\n"
        "The dialog demonstrates the use of the `on_frontend_error` plugin.",
    )
    error_button.on_click(lambda _: panel.app.commands.execute_method("execute", "Not a command"))
    panel.children = [
        ipw.HTML(
            f"<h3>{path}</h3> Welcome to my app.<br> kernel id: {panel.app.kernelId}<br>{cwd=}"
            "<br>Try the context menu with: right click -> Open console"
        ),
        error_button,
    ]

    # Do something when the window is closed (shutdown the kernel)
    def on_close(_):
        async def shutdown():
            result = await ipylab.app.dialog.show_dialog("Shutdown kernel?")
            if result["value"]:
                ipylab.app.notification.notify("Shutting down kernel", type=ipylab.NotificationType.info)
                ipylab.app.shutdown_kernel()

        ipylab.app.to_task(shutdown())

    # Add a plugin in this kernel
    class MyLocalPlugin:
        @ipylab.hookimpl
        def on_frontend_error(self, obj: ipylab.Ipylab, error: Exception, content: dict, buffers):  # noqa: ARG002
            ipylab.app.notification.notify("Clicked 'Do an error'", type=ipylab.NotificationType.error)

    app.plugin_manager.register(MyLocalPlugin())

    sc.observe(on_close, "comm")
    return panel


import ipylab  # noqa: E402


class MyPlugins:
    callcount = 0

    async def start_my_app(self, cwd: str, app: ipylab.JupyterFrontEnd):
        path = await app.dialog.get_text("Path for app")
        if not path:
            self.callcount += 1
            path = f"my app {self.callcount}"
        return await app.shell.add(create_app, cwd=cwd, path=path)

    @ipylab.hookimpl(specname="autostart")
    async def register_commands(self, app):
        cmd = await app.commands.add(
            "Start my app",
            execute=self.start_my_app,
            label="Start",
            caption="Start my custom app",
            icon_class="jp-PythonIcon",
        )
        await cmd.add_launcher("Ipylab")


pluginmodule = MyPlugins()

Instead of defining entry points and installing a module. Let's roughly simulate what the entry point does.

Although keep in mind that entry points will install the plugins for every kernel.

In [ ]:
# Loading the entry point
ipylab.app.plugin_manager.register(pluginmodule)

# When the ipylab kernel starts
ipylab.app._on_frontend_init()  # noqa: SLF001

We should now have a launcher

In [ ]:
t = ipylab.app.execute_command("launcher:create")